# Introduction to Data Analysis and Data Science

<img src="./img/1_data_analysis.jpeg" width="500px">

### Definitions


#### _Data Analysis:_

> &nbsp;  
> is the process of `inspecting`, `cleaning`, `transforming` and `modelling` data  
> with the goal to 
> - discover useful information, 
> - derive conclusions and 
> - support decission making.
> <br><br>

<span style="font-size: 70%">(quoted from [Wikipedia](https://en.wikipedia.org/wiki/Data_analysis), Original: "Transforming Unstructured Data into Useful Information", Big Data, Mining, and Analytics, Auerbach Publications, pp. 227–246, 2014-03-12)</span>
<br><br>

The process of data analysis comprises of:

1. define data requirements (based on research topic)
1. collect data (from selected data sources)
1. process (inspect, organize, structure) data
1. clean (segment, deduplicate, transform) data
1. explore (statistically, visually) data
1. develop models and algorithms
1. automate (data analysis)
1. communicate results

<br><br>
or visually:

<img src="./img/1_data_analysis_process.png">

Introductory reading: [Wikipedia - Data analysis](https://en.wikipedia.org/wiki/Data_analysis)


#### _Data Science:_

> &nbsp;  
> is the academic field that uses  
> - statistics, 
> - scientific computing and methods,
> - processes,
> - algorithms and systems  
>
> to extract or extrapolate knowledge and insights  
> from noisy, structured and unstructured data.
> <br><br>

<span style="font-size: 70%">(quoted from [Wikipedia](https://en.wikipedia.org/wiki/Data_science), Original: Dhar, V. (2013). "Data science and prediction". Communications of the ACM. 56 (12): pp. 64–73.)</span>
<br><br>

By this definition, data science is multidisciplinary, including `mathematics`, `statistics`, `computer` and `information science`.  
To fully exploit data science, `domain knowledge` is required to fully understand and analyse phenomena with data.
<br><br>

Data science covers a range of activities:

<img src="./img/1_data_science_process.png" width="72%">

<span style="font-size: 70%">If the colour scheme looks familiar to you, you may want to look into: [Intro to data science on Google Cloud](https://cloud.google.com/blog/topics/developers-practitioners/intro-data-science-google-cloud)</span>
<br><br>

Rule of thumb:

> &nbsp;  
> __Data analysis__ deals with practical application
> 
> __Data science__ covers the theory
> <br><br>


Introductory reading: [Wikipedia - Data science](https://en.wikipedia.org/wiki/Data_science)


<br><br><br>

# First real example

Assume a network of related people.

<img src="./img/1_friends.png" width="50%">

<span style="font-size: 70%">_Figure 1: Relationships_</span>
<br><br>

Let's model the data first.

In [ ]:
# we call the people 'users' to stay in line with the accompanying book

users = [
    {"id": 0, "name": "Hero"},
    {"id": 1, "name": "Dunn"},
    {"id": 2, "name": "Sue"},
    {"id": 3, "name": "Chi"},
    {"id": 4, "name": "Thor"},
    {"id": 5, "name": "Clive"},
    {"id": 6, "name": "Hicks"},
    {"id": 7, "name": "Devin"},
    {"id": 8, "name": "Kate"},
    {"id": 9, "name": "Klein"}
]

We could model the relationships like this.

In [ ]:
# there are several ways to model relations, this is a simple one

friendship_pairs = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
                    (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)]

Let's check:

__5__ is connected to __4__, __6__ and __7__.  
This is modeled by the tuples ... (4, 5), (5, 6), (5, 7) ...

In a real world problem, ___Clive___ would be friends with ___Thor___, ___Hicks___ and ___Devin___.

Now we want to generate some sort of lookup table that would return __all__ friends of a given person.

In [ ]:
# a dictionary is fast in looking up items.

# Initialize the dict with an empty list for each user id:
friendships = {user["id"]: [] for user in users}

# we can check on the value of 'friendships' by using Pythons REPL (read-eval-print loop) capabilities.
friendships

We have generated a dictionary with `keys` 0 ... 9 representing our users and empty lists as `values`.

Let's fill this dict's `values`.

In [ ]:
# And loop over the friendship pairs to populate it:

for i, j in friendship_pairs:
    friendships[i].append(j)                            # Add j as a friend of user i
    friendships[j].append(i)                            # Add i as a friend of user j

# check 'friendships'
friendships

In [ ]:
# Now we can check Clives friends directly.

friendships[5]

##### What is the average number of friends?

In [ ]:
# return number of friends a user has

def number_of_friends(user):
    """How many friends does _user_ have?"""
    user_id = user["id"]                                # extract the user id
    friend_ids = friendships[user_id]                   # extract all friends for a user
    return len(friend_ids)                              # count by returning the length of elements in the list

# this is a brief test for our function
assert number_of_friends({"id": 5, "name": "Clive"}) == 3, "Enter user as a dict of the list of users"

# get all connections
total_connections = sum(number_of_friends(user) for user in users)

int(total_connections / 2)                              # check number of edges in Figure 1

In [ ]:
# calculare the average number of connections

num_users = len(users)                                  # length of the users list
avg_connections = total_connections / num_users         # 24 / 10 == 2.4

print(f'The average number of connections is:  {avg_connections}')

##### Can we get a (reverse) ordered list of people with the number of their friends?

In [ ]:
# Create a list (user_id, number_of_friends)

num_friends_by_id = [(user["id"], number_of_friends(user)) for user in users]

num_friends_by_id.sort(                                 # Sort the list
       key=lambda id_and_friends: id_and_friends[1],    # by num_friends
       reverse=True)                                    # largest to smallest

assert num_friends_by_id[0][1] == 3                     # several people have 3 friends
assert num_friends_by_id[-1] == (9, 1)                  # user 9 has only 1 friend

# Each pair is (user_id, num_friends):
print(num_friends_by_id)

This can be visualized.
<br><br>

<img src="./img/1_friends_weighted.png" width="50%">

<span style="font-size: 70%">_Figure 2: Weighted Relationships_</span>
<br><br>

##### How do Xing and LinkedIn recommend new contacts?

We want to find the peers of the friends.

In [ ]:
# first attempt

def foaf_ids_1(user):
    """foaf is short for "friend of a friend" """
    return [foaf_id
            for friend_id in friendships[user["id"]]
            for foaf_id in friendships[friend_id]]


# test the result
foaf_ids_1({"id": 0, "name": "Hero"})


The result includes the users themselves (we don't want that),  
their friends (we don't want that either) and  
the peers twice (or multiple times, which we don't need).

For user Hero (id: 0) we want 3 as a recommendation, for user Clive (id: 5) we want [3, 8].  

We will write the assert statements first:

> &nbsp;  
> assert list(set(foaf_ids_2(users[0]))) == [3,]  
> assert list(set(foaf_ids_2(users[5]))) == [8, 3]
> <br><br>

In [ ]:
# eliminating user self and friends

def friends_of_friends(user):
    user_id = user["id"]
    return [foaf_id
            for friends_id in friendships[user_id]
            for foaf_id in friendships[friends_id]
            if foaf_id is not user_id                      # eliminate self
            and foaf_id not in friendships[user_id]]       # eliminate own friends

# test
assert list(set(friends_of_friends(users[0]))) == [3,]
assert list(set(friends_of_friends(users[5]))) == [8, 3]

# to eliminate duplicates we could use sets and cast to lists
print(f"Friends of friends of user {users[5]['name']}:  {friends_of_friends(users[5])}")
print(f"Eliminate duplicates:              {list(set(friends_of_friends(users[5])))}")

Now we can get ranked recommendations, be changing the return value to something suitable ...  

<span style="font-size: 70%">__Hint:__ If you are interested in a specific function or object, VSCode and Jupyter Lab provide online help.<br>
In _Jupyter Lab_ <cmd-I> provides contextual help. In _VSCode_ hovering over a keyword triggers tooltip-like help.</span>

In [ ]:
# for a recommendation system, Counter returns a ranked set (well, actually, it returns a Counter object)

from collections import Counter                             # redundant

def recommend_new_contacts(user):
    """recommends new peers based on friends of friends"""
    return Counter(friends_of_friends(user))                # eliminate own friends

# test correct result
assert recommend_new_contacts(users[7]) == Counter({4: 1, 6: 2, 9: 1})

print(
    f"Recommendations for user {users[5]['name']}:  " +
    f"{list(recommend_new_contacts(users[7]).keys())}")
print(f"Ranked recommendations:          {recommend_new_contacts(users[7])}")


... and can use the information to find users with mutual friends.

In [ ]:
# find mutual friends

u1 = users[0]
u2 = users[3]

# this can be done more elegantly
if u1['id'] in recommend_new_contacts(u2):
    print(f"{users[0]['name']} and {users[3]['name']} have mutual friends!")
else:
    print(f"{users[0]['name']} and {users[3]['name']} share no mutual friends!")

##### Find people with similar interest

In [ ]:
# fields of interest
# data structure: (user_id: int, interest: string)

interests = [
    (0, "Hadoop"), (0, "Big Data"), (0, "HBase"), (0, "Java"),
    (0, "Spark"), (0, "Storm"), (0, "Cassandra"),
    (1, "NoSQL"), (1, "MongoDB"), (1, "Cassandra"), (1, "HBase"),
    (1, "Postgres"), (2, "Python"), (2, "scikit-learn"), (2, "scipy"),
    (2, "numpy"), (2, "statsmodels"), (2, "pandas"), (3, "R"), (3, "Python"),
    (3, "statistics"), (3, "regression"), (3, "probability"),
    (4, "machine learning"), (4, "regression"), (4, "decision trees"),
    (4, "libsvm"), (5, "Python"), (5, "R"), (5, "Java"), (5, "C++"),
    (5, "Haskell"), (5, "programming languages"), (6, "statistics"),
    (6, "probability"), (6, "mathematics"), (6, "theory"),
    (7, "machine learning"), (7, "scikit-learn"), (7, "Mahout"),
    (7, "neural networks"), (8, "neural networks"), (8, "deep learning"),
    (8, "Big Data"), (8, "artificial intelligence"), (9, "Hadoop"),
    (9, "Java"), (9, "MapReduce"), (9, "Big Data")
]

In [ ]:
def data_scientists_who_like(target_interest):
    """Find the ids of all users who like the target interest."""
    return [user_id
            for user_id, user_interest in interests
            if user_interest == target_interest]

# test
assert data_scientists_who_like('Python') == [2, 3, 5]
assert data_scientists_who_like('Java') == [0, 5, 9]

For large lists this is inefficient.

Let's build lookup tables

In [ ]:
# lookup tables based on defaultdict (a dict that creates new values on the fly, similar to dict.setdefault() )

from collections import defaultdict

# Keys are interests, values are lists of user_ids with that interest
user_ids_by_interest = defaultdict(list)

for user_id, interest in interests:
    user_ids_by_interest[interest].append(user_id)

# Keys are user_ids, values are lists of interests for that user_id.
interests_by_user_id = defaultdict(list)

for user_id, interest in interests:
    interests_by_user_id[user_id].append(interest)

print(f"Users by interest: \n{user_ids_by_interest}\n")
print(f"Interests by users: \n{interests_by_user_id}")

In [ ]:
# we can look up user names while creating the list

interests_by_users = defaultdict(list)

for id, interest in interests:
    interests_by_users[users[id]['name']].append(interest)

print(f"Users (name) by interest: \n{interests_by_users}")

In [ ]:
# find the most common interests

def most_common_interests_with(user):
    return Counter(
        interested_user_id
        for interest in interests_by_user_id[user["id"]]
        for interested_user_id in user_ids_by_interest[interest]
        if interested_user_id != user["id"]
    )

most_common_interests_with(users[0])

##### Find the most popular topics

In [ ]:
# not elegant but does the job

words_and_counts = Counter(word
                           for user, interest in interests
                           for word in [interest.lower(), ])    # return a list instead of string

for word, count in words_and_counts.most_common():
    if count > 1:
        print(f"{word}: {count}")


<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>Can you rewrite "most_common_interest_with(user) that returns user names instead of user_id?</td>
</tr>
</table>

In [ ]:
# with user name ... students attempt

def most_common_interests_by_name(user):
    pass


assert most_common_interests_by_name(users[0]) == Counter({'Klein': 3, 'Dunn': 2, 'Kate': 1, 'Clive': 1})


In [ ]:
# with user name ... Solution - do not unfold before students attempts

def most_common_interests_by_name(user):
    return Counter(
        users[interested_user_id]['name']
        for interest in interests_by_user_id[user["id"]]
        for interested_user_id in user_ids_by_interest[interest]
        if interested_user_id != user["id"]
    )

most_common_interests_by_name(users[0])


<br><br><br>

# Analyzing salaries

Assume we have a data set consisting of yearly salaries and tenure (time of employment in the company).

Our data set is a list of tupes: [ ( \<salary>, \<tenure>), ( \<salary>, \<tenure>), ... ]

In [ ]:
# here is the data

salaries_and_tenures = [(83000, 8.7), (88000, 8.1),
                        (48000, 0.7), (76000, 6),
                        (69000, 6.5), (76000, 7.5),
                        (60000, 2.5), (83000, 10),
                        (48000, 1.9), (63000, 4.2)]

In [ ]:
# visualize data (without any fuzz)

import matplotlib.pyplot as plt

plt.scatter([x[1] for x in salaries_and_tenures], [y[0]
            for y in salaries_and_tenures])
plt.show()


<span style="font-size: 70%">_Figure 3: Salary per tenure duration_. More on `mathplotlib` in Chapter 3</span>
<br><br>

The graph can be easily interpreted as: _The longer an employee works for a company, the higher his salary._

##### Average salary

In [ ]:
# a single average is easy to calculate but does not convey much information

salaries = [y[0] for y in salaries_and_tenures]
avg_salary = sum(salaries) / len(salaries)
avg_salary


In [ ]:
# putting values into buckets and computing the average for each bucket makes more sense

# bucket sizes:
#   0 - 2 years
#   2 - 5 years
#   5+    years

NEW = 2
INTERMEDIATE = 5
EXPERT = 7

def tenure_bucket(tenure):
    if tenure < NEW:
        return "less than two"
    elif tenure < INTERMEDIATE:
        return "between two and five"
    else:
        return "more than five"

# Keys are tenure buckets, values are lists of salaries for that bucket.
salary_by_tenure_bucket = defaultdict(list)

for salary, tenure in salaries_and_tenures:
    bucket = tenure_bucket(tenure)
    salary_by_tenure_bucket[bucket].append(salary)

# Keys are tenure buckets, values are average salary for that bucket
average_salary_by_bucket = {
    tenure_bucket: sum(salaries) / len(salaries)
    for tenure_bucket, salaries in salary_by_tenure_bucket.items()
}

average_salary_by_bucket

In [ ]:
delta_percent = (average_salary_by_bucket['more than five'] - average_salary_by_bucket['less than two']) / average_salary_by_bucket['less than two'] * 100

print(f"Average initial salary is: {average_salary_by_bucket['less than two']}")
print(f"Senior employees earn  {delta_percent:2.0f}%  more than freshmen.")


Our buckets were chosen quite randomly (gut feeling).
<br><br>
<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>Experiment with the bucket boundaries.<br>Extend "tenure_bucket()" with a fourth bucket.</td>
</tr>
</table>

<br><br><span style="font-size: 128px">&#9749;</span> Coffee break!